# 04 - Build ChromaDB Regulatory Knowledge Base

**ITAI 2376 Final Project - GridGuard-AI**  
**Authors:** DeMarcus Crump & Yoana Cook

Indexes the 28 NERC / ERCOT / FERC regulatory PDFs in `data/regulatory_docs/` into a local **ChromaDB** vector store that the Regulatory Knowledge agent queries at runtime via semantic similarity (RAG).

### Pipeline
1. Walk `data/regulatory_docs/` and extract text from every PDF with `pypdf`.
2. Chunk each document into ~800-token passages with 120-token overlap.
3. Embed each chunk using `sentence-transformers/all-MiniLM-L6-v2` (local, free, 384-dim).
4. Persist the collection to `data/chroma_db/` - this folder is committed to git so the runtime agent does not have to re-embed on every cold-start.

### Deep-learning connection
The embedding model is a **Transformer** (MiniLM) fine-tuned for sentence similarity. Each regulatory clause becomes a dense vector, and retrieval is nearest-neighbour cosine similarity - direct application of the embeddings and attention-weighted representation concepts from Module 05.

Expected runtime in Colab: ~3-5 min (depends on PDF page count).

In [ ]:
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip -q install chromadb==0.4.24 sentence-transformers==2.7.0 pypdf==4.2.0
    REPO_URL = os.environ.get('GRIDGUARD_REPO_URL', 'https://github.com/<YOUR_GITHUB_USER>/Crump_GridGuard_ITAI2376.git')
    if not os.path.isdir('/content/repo'):
        !git clone {REPO_URL} /content/repo
    os.chdir('/content/repo')
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')
print('Working directory:', os.getcwd())

In [ ]:
import glob, re, json, hashlib
from pathlib import Path
from pypdf import PdfReader
import chromadb
from chromadb.utils import embedding_functions

PDF_ROOT    = Path('data/regulatory_docs')
CHROMA_DIR  = Path('data/chroma_db')
COLLECTION  = 'gridguard_regulatory_kb'
CHUNK_WORDS = 500         # ~800 tokens
OVERLAP     = 80

CHROMA_DIR.mkdir(parents=True, exist_ok=True)

pdfs = sorted(PDF_ROOT.rglob('*.pdf'))
print(f'Found {len(pdfs)} PDFs under {PDF_ROOT}')
for p in pdfs:
    print(f'  {p.relative_to(PDF_ROOT)}')

In [ ]:
# --- Text extraction + chunking ---
def extract_text(pdf_path: Path) -> str:
    try:
        reader = PdfReader(str(pdf_path))
        pages = [p.extract_text() or '' for p in reader.pages]
        text  = '\n'.join(pages)
        text  = re.sub(r'\s+', ' ', text).strip()
        return text
    except Exception as exc:
        print(f'  [WARN] failed to parse {pdf_path.name}: {exc}')
        return ''

def chunk(text: str, size: int = CHUNK_WORDS, overlap: int = OVERLAP):
    words = text.split()
    if not words:
        return []
    chunks, step = [], size - overlap
    for i in range(0, len(words), step):
        piece = ' '.join(words[i:i+size])
        if len(piece) > 80:
            chunks.append(piece)
        if i + size >= len(words):
            break
    return chunks

corpus = []   # list of (id, text, metadata)
for pdf in pdfs:
    category = pdf.parent.name
    source   = pdf.name
    text     = extract_text(pdf)
    for idx, piece in enumerate(chunk(text)):
        uid = hashlib.md5(f'{source}-{idx}'.encode()).hexdigest()
        corpus.append((uid, piece, {'source': source, 'category': category, 'chunk_index': idx}))

print(f'\nTotal chunks across {len(pdfs)} PDFs: {len(corpus):,}')

In [ ]:
# --- Build / replace the ChromaDB collection ---
client = chromadb.PersistentClient(path=str(CHROMA_DIR))
try:
    client.delete_collection(COLLECTION)
    print(f'Deleted existing collection "{COLLECTION}".')
except Exception:
    pass

embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)
collection = client.create_collection(
    name=COLLECTION,
    embedding_function=embed_fn,
    metadata={'hnsw:space': 'cosine'},
)

# Batched upsert (Chroma recommends <= 5000 per call)
BATCH = 256
for start in range(0, len(corpus), BATCH):
    batch = corpus[start:start+BATCH]
    collection.add(
        ids       = [row[0] for row in batch],
        documents = [row[1] for row in batch],
        metadatas = [row[2] for row in batch],
    )
    print(f'  embedded {min(start+BATCH, len(corpus)):>6,} / {len(corpus):,}')

print(f'\nCollection "{COLLECTION}" now holds {collection.count():,} chunks.')

In [ ]:
# --- Sanity-check the RAG retrieval ---
PROBE_QUERIES = [
    'What is the reserve margin threshold that triggers EEA Level 2?',
    'NERC BAL-001 balancing authority requirements',
    'Uri 2021 winter storm generator weatherization lessons',
    'ERCOT system operating limit methodology',
]

for q in PROBE_QUERIES:
    res = collection.query(query_texts=[q], n_results=3)
    print(f'\nQ: {q}')
    for doc, meta, dist in zip(res['documents'][0], res['metadatas'][0], res['distances'][0]):
        print(f'  [{dist:.3f}] {meta["source"]:<42} {doc[:120].strip()}...')

In [ ]:
# --- Write a small manifest so the runtime agent knows what it is loading ---
manifest = {
    'collection'   : COLLECTION,
    'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
    'n_documents'  : len(pdfs),
    'n_chunks'     : collection.count(),
    'chunk_words'  : CHUNK_WORDS,
    'chunk_overlap': OVERLAP,
    'source_categories': sorted({p.parent.name for p in pdfs}),
    'documents'    : [p.name for p in pdfs],
}
with open(CHROMA_DIR / 'manifest.json', 'w') as fh:
    json.dump(manifest, fh, indent=2)

print('Chroma store persisted to', CHROMA_DIR)
print('Manifest:', json.dumps(manifest, indent=2))

if IN_COLAB:
    !zip -r -q chroma_db.zip data/chroma_db
    from google.colab import files
    files.download('chroma_db.zip')
    print('Downloaded chroma_db.zip. Unzip it into your repo so data/chroma_db/ is populated, then commit.')